In [ ]:
from sentence_transformers import SentenceTransformer, util
from pathlib import Path
import pandas as pd
import glob

In [ ]:
#model_name = "sentence-transformers/LaBSE"
model_name = "sentence-transformers/all-MiniLM-L12-v2"

model = SentenceTransformer(model_name)

In [ ]:
def evaluate_sentence_similarity(df):
    embeddings_ref = model.encode(df["translation_reference"].tolist(), convert_to_tensor=True, batch_size=32)
    embeddings_hyp = model.encode(df["translation_candidate"].tolist(), convert_to_tensor=True, batch_size=32)

    cosine_scores = util.cos_sim(embeddings_ref, embeddings_hyp)
    df["similarity_score"] = [float(cosine_scores[i][i]) for i in range(len(df))]

    return df["similarity_score"].mean()

In [ ]:
def get_working_df(df_references, df_candidate):
    df_final = pd.merge(
        df_references,
        df_candidate,
        on=['id', 'input'],
        how='inner'
    )

    return df_final[df_final['translation_reference'].notna()]

In [ ]:
# Configure data paths
TRANSLATIONS_DIR = "path/to/translations"  # Directory containing translation .tsv files
REFERENCES_FILE = "path/to/gold_reference.xlsx"  # Excel file with gold standard translations

df_references = pd.read_excel(REFERENCES_FILE).rename(columns={'translation': 'translation_reference'})

for file in glob.glob(TRANSLATIONS_DIR + "/*.tsv"):
    df_candidate = pd.read_csv(file, sep="\t").rename(columns={'translation': 'translation_candidate'})

    df_evaluation = get_working_df(df_references, df_candidate)

    result = evaluate_sentence_similarity(df_evaluation)
    print(50*"-" + "\n" + f"Translation results for {file}: ")
    print("Number of sentences:", len(df_evaluation))
    print(f"Average similarity: {result:.4f}")